# Car Price Prediction with Machine Learning

# Import Required Libraries

In [3]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

 # CREATE A SYNTHETIC DATASET

In [5]:
print("--- Step 1: Generating Dataset ---")
np.random.seed(42)
n_cars = 300

data = {
    "Year": np.random.randint(2012, 2025, n_cars),
    "Mileage": np.random.randint(5000, 140000, n_cars),
    "Engine_Size": np.round(np.random.uniform(1.0, 3.5, n_cars), 1),
    "Fuel_Type": np.random.choice(["Petrol", "Diesel", "Electric"], n_cars),
    "Transmission": np.random.choice(["Manual", "Automatic"], n_cars),
}

df = pd.DataFrame(data)

# Calculate a realistic target price based on features with random noise
df["Price"] = (
    32000
    - ((2026 - df["Year"]) * 1400)
    - (df["Mileage"] * 0.06)
    + (df["Engine_Size"] * 2200)
)
df.loc[df["Fuel_Type"] == "Electric", "Price"] += 8000
df.loc[df["Transmission"] == "Automatic", "Price"] += 1800
df["Price"] = df["Price"].clip(lower=3000).astype(int)

# Save to a local CSV file
df.to_csv("car_dataset.csv", index=False)
print("Saved 300 rows of vehicle records to 'car_dataset.csv'.\n")




--- Step 1: Generating Dataset ---
Saved 300 rows of vehicle records to 'car_dataset.csv'.



# LOAD AND PREPROCESS DATA

In [6]:
print("--- Step 2: Loading & Preprocessing Data ---")
df_loaded = pd.read_csv("car_dataset.csv")

# Feature Engineering: Convert calendar year into car age
df_loaded["Age"] = 2026 - df_loaded["Year"]
df_loaded = df_loaded.drop("Year", axis=1)

# Categorical Encoding: Convert words (text) into binary 1s and 0s
df_encoded = pd.get_dummies(df_loaded, columns=["Fuel_Type", "Transmission"])
df_encoded = df_encoded.astype(int)

print("Preprocessed data columns:")
print(list(df_encoded.columns))
print("")



--- Step 2: Loading & Preprocessing Data ---
Preprocessed data columns:
['Mileage', 'Engine_Size', 'Price', 'Age', 'Fuel_Type_Diesel', 'Fuel_Type_Electric', 'Fuel_Type_Petrol', 'Transmission_Automatic', 'Transmission_Manual']



# SEPARATE FEATURES AND TARGET SPLIT

In [7]:
print("--- Step 3: Splitting Dataset ---")
# Features (Inputs)
X = df_encoded.drop("Price", axis=1)
# Target (Output we want to predict)
y = df_encoded["Price"]

# Split data: 80% for training the model, 20% reserved for testing accuracy
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training Features Shape: {X_train.shape}")
print(f"Testing Features Shape: {X_test.shape}\n")


--- Step 3: Splitting Dataset ---
Training Features Shape: (240, 8)
Testing Features Shape: (60, 8)



# INITIALIZE AND TRAIN THE MACHINE LEARNING MODEL

In [8]:
print("--- Step 4: Training Random Forest Regressor ---")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("Model training complete.\n")


--- Step 4: Training Random Forest Regressor ---
Model training complete.



# EVALUATE PERFORMANCES

In [9]:
print("--- Step 5: Model Evaluation Metrics ---")
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Model R-squared Accuracy (R2 Score): {r2 * 100:.2f}%\n")

--- Step 5: Model Evaluation Metrics ---
Mean Absolute Error (MAE): $1383.57
Model R-squared Accuracy (R2 Score): 94.88%



# PREDICT VALUE FOR A USER DEFINED CUSTOM CAR

In [10]:
print("--- Step 6: Custom Pricing Prediction ---")


def estimate_car_price(mileage, engine_size, age, fuel, transmission):
    # Match the exact column format expected by our model
    custom_input = pd.DataFrame(0, index=[0], columns=X.columns)

    # Assign raw values
    custom_input["Mileage"] = mileage
    custom_input["Engine_Size"] = engine_size
    custom_input["Age"] = age

    # Map text labels to active encoded columns
    if f"Fuel_Type_{fuel}" in custom_input.columns:
        custom_input[f"Fuel_Type_{fuel}"] = 1
    if f"Transmission_{transmission}" in custom_input.columns:
        custom_input[f"Transmission_{transmission}"] = 1

    # Return prediction
    predicted_val = model.predict(custom_input)[0]
    return predicted_val


# Input parameters: Mileage, Engine Size (L), Age (Years), Fuel, Transmission
sample_mileage = 40000
sample_engine = 2.0
sample_age = 3
sample_fuel = "Petrol"
sample_transmission = "Automatic"

final_valuation = estimate_car_price(
    mileage=sample_mileage,
    engine_size=sample_engine,
    age=sample_age,
    fuel=sample_fuel,
    transmission=sample_transmission,
)

print(f"Target Vehicle Specs: {sample_age} yrs old, {sample_mileage} miles, "
      f"{sample_engine}L Engine, {sample_fuel}, {sample_transmission}")
print(f"--> Estimated Market Price: ${final_valuation:,.2f}")

--- Step 6: Custom Pricing Prediction ---
Target Vehicle Specs: 3 yrs old, 40000 miles, 2.0L Engine, Petrol, Automatic
--> Estimated Market Price: $31,471.01


 # EVALUATE MODEL PERFORMANCE METRICS

In [11]:




print("--- Step 7: System Validation & Performance Metrics ---")
from sklearn.metrics import mean_squared_error

# Make predictions on the hidden testing data block (20% split)
y_pred = model.predict(X_test)

# Calculate key model accuracy indicators
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("------------------------------------------")
print(f" Mean Absolute Error (MAE)   : ${mae:.2f} (Average margin of error)")
print(f" Root Mean Squared Error(RMSE): ${rmse:.2f} (Penalizes larger errors)")
print(f" Overall Accuracy Score (R2)  : {r2 * 100:.2f}%")
print("------------------------------------------\n")



--- Step 7: System Validation & Performance Metrics ---
------------------------------------------
 Mean Absolute Error (MAE)   : $1383.57 (Average margin of error)
 Root Mean Squared Error(RMSE): $1681.98 (Penalizes larger errors)
 Overall Accuracy Score (R2)  : 94.88%
------------------------------------------



# MODEL SERIALIZATION & DEPLOYMENT PREPARATION 

In [12]:

print("--- Step 8: Freezing and Saving Model Pipeline ---")
import joblib

# Export your trained model weights to a file so it can load instantly later
model_filename = "car_price_predictor.pkl"
joblib.dump(model, model_filename)

print(f"Success: Machine learning model successfully saved as '{model_filename}'!")
print("You can now build a web app or dashboard that loads this file directly.")


--- Step 8: Freezing and Saving Model Pipeline ---
Success: Machine learning model successfully saved as 'car_price_predictor.pkl'!
You can now build a web app or dashboard that loads this file directly.
